## Loading Data and Pre-processing

In [ ]:
import pandas as pd
import numpy as np
from numpy.random import normal

import re

In [ ]:
np.random.seed(123)

In [ ]:
data = pd.read_excel("nlp_sentiment_data.xlsx")

NameError: ignored

In [ ]:
data.head()

NameError: ignored

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4641 entries, 0 to 4640
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              4641 non-null   int64         
 1   SOURCE          4638 non-null   object        
 2   REVIEW BY       4640 non-null   object        
 3   REVIEW DATE     4641 non-null   datetime64[ns]
 4   REVIEW SUBJECT  2766 non-null   object        
 5   text            4637 non-null   object        
 6   REVIEW RATING   4641 non-null   int64         
 7   REVIEW TYPE     4641 non-null   object        
 8   sentiment       4641 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 326.4+ KB


In [ ]:
data = data[['text','sentiment']]
data_no_nan = data.dropna()

data_no_nan.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4637 entries, 0 to 4640
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       4637 non-null   object
 1   sentiment  4637 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 108.7+ KB


In [ ]:
data_no_nan['sentiment'].value_counts() / len(data_no_nan)

1    0.882251
0    0.117749
Name: sentiment, dtype: float64

## Training and Validation splitting of dataset.

In [ ]:
from sklearn.model_selection import train_test_split
train, valid = train_test_split(data_no_nan, test_size=0.1, random_state=123)

train_text = train.iloc[:, 0]
train_class = train.iloc[:, 1]

valid_text = valid.iloc[:, 0]
valid_class = valid.iloc[:, 1]

In [ ]:
train.iloc[:, 1].value_counts()/len(train)

1    0.88186
0    0.11814
Name: sentiment, dtype: float64

In [ ]:
valid.iloc[:, 1].value_counts()/len(valid)

1    0.885776
0    0.114224
Name: sentiment, dtype: float64

## Handling Imbalanced Data

In [ ]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=123)

X_resampled, y_resampled = ros.fit_resample(train[['text']], train['sentiment'])

from collections import Counter
print(sorted(Counter(y_resampled).items()))

train_resampled = pd.concat([pd.Series(np.squeeze(X_resampled, axis=1)),
                             pd.Series(y_resampled)], axis=1)

# Training and Validation Dataset -- For resampled dataset
train_text = train_resampled.iloc[:, 0]
train_class = train_resampled.iloc[:, 1]

[(0, 3680), (1, 3680)]


/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


## Tokenization, Vocabulary & Sequence Creation

In [ ]:
from tensorflow.keras.preprocessing import sequence, text

# Vectorization parameters
# Limit on the number of features (Vocabulary). We use the top 3K features.
TOP_K = 3000

In [ ]:
lens = np.array(list(map(len, train_text)))
(lens.max(), lens.min(), lens.mean())

(4467, 2, 247.49809782608696)

In [ ]:
# Limit on the length of text sequences. Sequences longer than this
# will be truncated.
MAX_SEQUENCE_LENGTH = 250

In [ ]:
# Create vocabulary with training texts.
tokenizer = text.Tokenizer(num_words=TOP_K)
tokenizer.fit_on_texts(train_text)

'\nUpdates internal vocabulary based on a list of texts.\nIn the case where texts contains lists, we assume each entry of the lists to be a token.\nRequired before using texts_to_sequences or texts_to_matrix.\n'

In [ ]:
# Converting training and validation texts to sequences of integers.

x_train = tokenizer.texts_to_sequences(train_text)
x_val = tokenizer.texts_to_sequences(valid_text)

In [ ]:
print(x_train[0:1])

[[271, 370, 1, 592, 439, 213]]


In [ ]:
# Fixing the sentence size to max sequence length.
x_train = sequence.pad_sequences(x_train, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
x_val = sequence.pad_sequences(x_val, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

In [ ]:
print(x_train[0:1])

[[271 370   1 592 439 213   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0 

In [ ]:
x_train.shape

(7360, 250)

## Model Building using Embedding layer and Fully Connected layer

In [ ]:
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential

In [ ]:
model = Sequential([
    Embedding(TOP_K, 32, input_length=MAX_SEQUENCE_LENGTH),
    BatchNormalization(),
    Flatten(),
    Dense(10, activation='relu'),
    Dropout(rate=0.7),
    Dense(1, activation='sigmoid')])

In [ ]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_3 (Embedding)      (None, 250, 32)           96000     
_________________________________________________________________
batch_normalization_3 (Batch (None, 250, 32)           128       
_________________________________________________________________
flatten_3 (Flatten)          (None, 8000)              0         
_________________________________________________________________
dense_6 (Dense)              (None, 10)                80010     
_________________________________________________________________
dropout_3 (Dropout)          (None, 10)                0         
_________________________________________________________________
dense_7 (Dense)              (None, 1)                 11        
Total params: 176,149
Trainable params: 176,085
Non-trainable params: 64
_______________________________________________

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(x_train, train_class, validation_data=(x_val, valid_class), epochs=5, batch_size=64)

Epoch 1/5
115/115 [==============================] - 1s 6ms/step - loss: 0.6506 - accuracy: 0.6264 - val_loss: 0.7186 - val_accuracy: 0.1250
Epoch 2/5
115/115 [==============================] - 1s 5ms/step - loss: 0.4293 - accuracy: 0.8365 - val_loss: 0.5538 - val_accuracy: 0.8599
Epoch 3/5
115/115 [==============================] - 1s 5ms/step - loss: 0.2656 - accuracy: 0.9281 - val_loss: 0.3038 - val_accuracy: 0.8879
Epoch 4/5
115/115 [==============================] - 1s 5ms/step - loss: 0.2010 - accuracy: 0.9514 - val_loss: 0.3010 - val_accuracy: 0.8944
Epoch 5/5
115/115 [==============================] - 1s 5ms/step - loss: 0.1857 - accuracy: 0.9550 - val_loss: 0.4333 - val_accuracy: 0.8901


## Using Pre-trained Embeddings

First, we would need to upload the glove vectors into the instance.

In [ ]:
# Create a directory 'glove'
!mkdir -p ./glove

In [ ]:
# Unzip
!unzip ./glove_6B_50d.zip -d ./glove/

Archive:  ./glove_6B_50d.zip
  inflating: ./glove/glove_6B_50d.txt  


In [ ]:
# After uncompressing, we get a text file.
!ls -lah ./glove/

total 164M
drwxr-xr-x 2 root root 4.0K Dec 12 05:25 .
drwxr-xr-x 1 root root 4.0K Dec 12 05:04 ..
-rw-r--r-- 1 root root 164M Aug  5  2014 glove_6B_50d.txt


In [ ]:
# The text file has one line for a character/word and its corresponding vector.
!head -5 ./glove/glove_6B_50d.txt

the 0.418 0.24968 -0.41242 0.1217 0.34527 -0.044457 -0.49688 -0.17862 -0.00066023 -0.6566 0.27843 -0.14767 -0.55677 0.14658 -0.0095095 0.011658 0.10204 -0.12792 -0.8443 -0.12181 -0.016801 -0.33279 -0.1552 -0.23131 -0.19181 -1.8823 -0.76746 0.099051 -0.42125 -0.19526 4.0071 -0.18594 -0.52287 -0.31681 0.00059213 0.0074449 0.17778 -0.15897 0.012041 -0.054223 -0.29871 -0.15749 -0.34758 -0.045637 -0.44251 0.18785 0.0027849 -0.18411 -0.11514 -0.78581
, 0.013441 0.23682 -0.16899 0.40951 0.63812 0.47709 -0.42852 -0.55641 -0.364 -0.23938 0.13001 -0.063734 -0.39575 -0.48162 0.23291 0.090201 -0.13324 0.078639 -0.41634 -0.15428 0.10068 0.48891 0.31226 -0.1252 -0.037512 -1.5179 0.12612 -0.02442 -0.042961 -0.28351 3.5416 -0.11956 -0.014533 -0.1499 0.21864 -0.33412 -0.13872 0.31806 0.70358 0.44858 -0.080262 0.63003 0.32111 -0.46765 0.22786 0.36034 -0.37818 -0.56657 0.044691 0.30392
. 0.15164 0.30177 -0.16763 0.17684 0.31719 0.33973 -0.43478 -0.31086 -0.44999 -0.29486 0.16608 0.11963 -0.41328 -0.42353

In [ ]:
path = './glove/'

In [ ]:
!pip install bcolz # It is a library used for efficient compression.

     |████████████████████████████████| 1.5MB 8.7MB/s 
  Created wheel for bcolz: filename=bcolz-1.2.1-cp36-cp36m-linux_x86_64.whl size=2659697 sha256=ad16c0f2415c696806eb127a3ed791f2367199c09223483c8f9795debf450c45
  Stored in directory: /root/.cache/pip/wheels/9f/78/26/fb8c0acb91a100dc8914bf236c4eaa4b207cb876893c40b745
Successfully built bcolz


In [ ]:
import bcolz, pickle

In [ ]:
# This function compresses the array using bcolz and this is called by get_glove function shown below.
def save_array(arr, filename):
    c = bcolz.carray(arr, rootdir=filename, mode='w')
    c.flush()

In [ ]:
# This function reads the unzipped glove file and extracts
# the words and vectors list and creates a word to index dictionary and compresses
# and stores them differently.

def get_glove(name):
    with open(path + 'glove_' + name + '.txt', encoding="utf8") as f:
        lines = [line.split() for line in f]

    words = [d[0] for d in lines]
    vecs = np.stack(np.array(d[1:], dtype=np.float32) for d in lines)
    wordidx = {o:i for i,o in enumerate(words)}

    # Save the vectors
    save_array(vecs, path+name+'_dat')
    # Save the words
    pickle.dump(words, open(path+name+'_words.pkl','wb'))
    # Save the word to index mapping
    pickle.dump(wordidx, open(path+name+'_idx.pkl','wb'))

In [ ]:
# We call the get_glove function by passing the 50 dimensional representation
# of the words.
get_glove('6B_50d')

/usr/local/lib/python3.6/dist-packages/IPython/core/interactiveshell.py:2882: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
# Uncompress the file compressed using bcolz
def load_array(filename):
    return bcolz.open(filename)[:]

In [ ]:
# This function returns the uncompressed words, vectors and the word to
# index dictionary.
def load_vectors(loc):
    return (load_array(loc+'_dat'),
        pickle.load(open(loc+'_words.pkl','rb')),
        pickle.load(open(loc+'_idx.pkl','rb')))

In [ ]:
vecs, words, wordidx = load_vectors(path+'6B_50d')

In [ ]:
# The output tuple shows that the
# first element of the words list is the word 'the' and the first element of
# the vectors list is the array of 50 numbers, corresponding to the word 'the'.

words[0], vecs[0]

('the', array([ 4.1800e-01,  2.4968e-01, -4.1242e-01,  1.2170e-01,  3.4527e-01,
        -4.4457e-02, -4.9688e-01, -1.7862e-01, -6.6023e-04, -6.5660e-01,
         2.7843e-01, -1.4767e-01, -5.5677e-01,  1.4658e-01, -9.5095e-03,
         1.1658e-02,  1.0204e-01, -1.2792e-01, -8.4430e-01, -1.2181e-01,
        -1.6801e-02, -3.3279e-01, -1.5520e-01, -2.3131e-01, -1.9181e-01,
        -1.8823e+00, -7.6746e-01,  9.9051e-02, -4.2125e-01, -1.9526e-01,
         4.0071e+00, -1.8594e-01, -5.2287e-01, -3.1681e-01,  5.9213e-04,
         7.4449e-03,  1.7778e-01, -1.5897e-01,  1.2041e-02, -5.4223e-02,
        -2.9871e-01, -1.5749e-01, -3.4758e-01, -4.5637e-02, -4.4251e-01,
         1.8785e-01,  2.7849e-03, -1.8411e-01, -1.1514e-01, -7.8581e-01],
       dtype=float32))

In [ ]:
words[1], vecs[1]

(',', array([ 0.013441,  0.23682 , -0.16899 ,  0.40951 ,  0.63812 ,  0.47709 ,
        -0.42852 , -0.55641 , -0.364   , -0.23938 ,  0.13001 , -0.063734,
        -0.39575 , -0.48162 ,  0.23291 ,  0.090201, -0.13324 ,  0.078639,
        -0.41634 , -0.15428 ,  0.10068 ,  0.48891 ,  0.31226 , -0.1252  ,
        -0.037512, -1.5179  ,  0.12612 , -0.02442 , -0.042961, -0.28351 ,
         3.5416  , -0.11956 , -0.014533, -0.1499  ,  0.21864 , -0.33412 ,
        -0.13872 ,  0.31806 ,  0.70358 ,  0.44858 , -0.080262,  0.63003 ,
         0.32111 , -0.46765 ,  0.22786 ,  0.36034 , -0.37818 , -0.56657 ,
         0.044691,  0.30392 ], dtype=float32))

We see that the second item in the glove is 'comma'.

Let us see the order in which the words are stored in the tokenizer built using our dataset.

In [ ]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'a': 4,
 'is': 5,
 'temple': 6,
 'of': 7,
 'you': 8,
 'in': 9,
 'place': 10,
 'it': 11,
 'i': 12,
 'for': 13,
 'this': 14,
 'very': 15,
 'are': 16,
 'krishna': 17,
 'visit': 18,
 'but': 19,
 'with': 20,
 'there': 21,
 'not': 22,
 'good': 23,
 'that': 24,
 'have': 25,
 'was': 26,
 'on': 27,
 'hare': 28,
 'one': 29,
 'be': 30,
 'bangalore': 31,
 'can': 32,
 'at': 33,
 'as': 34,
 'iskcon': 35,
 'like': 36,
 'all': 37,
 'well': 38,
 'food': 39,
 'if': 40,
 'they': 41,
 'will': 42,
 'we': 43,
 'nice': 44,
 'which': 45,
 'from': 46,
 'beautiful': 47,
 'time': 48,
 'more': 49,
 'its': 50,
 'also': 51,
 'go': 52,
 'so': 53,
 'people': 54,
 'inside': 55,
 'get': 56,
 'by': 57,
 'my': 58,
 'or': 59,
 'has': 60,
 'must': 61,
 'clean': 62,
 'visited': 63,
 'experience': 64,
 'main': 65,
 'your': 66,
 'see': 67,
 'lord': 68,
 'when': 69,
 'temples': 70,
 'through': 71,
 'commercial': 72,
 'other': 73,
 'no': 74,
 'where': 75,
 'many': 76,
 'feel': 77,
 'maintained':

We see that the order of the words are different from that of the glove list.

Hence we cannot directly fetch the vector values based on the index, rather we would need to search for the word from the glove vector and assign its corresponding vector to the word in the tokenizer.

We create an embedding matrix of dimension -- vocabulary size by embedding dimension size. And then we fetch the words from the tokenizer.index_word dictionary and search for the word in the glove's wordidx dictionary. If the word is available, we assign the vector values for the word else we randomly initialize the word.

In [ ]:
def create_emb():
    n_fact = vecs.shape[1]
    emb = np.zeros((TOP_K, n_fact))

    for i in range(1,len(emb)):
        word = tokenizer.index_word[i]
        if word and re.match(r"^[a-zA-Z0-9\-]*$", word) and word in wordidx:
            src_idx = wordidx[word]
            emb[i] = vecs[src_idx]
        else:
            # If we can't find the word in glove, randomly initialize
            emb[i] = normal(scale=0.6, size=(n_fact,))

    return emb

In [ ]:
emb = create_emb()

### Model Definition

The model has the embedding layer as the first layer, and we pass the embedding dimension size as 50 because glove has 50 numbers to represent a word. To ensure we use the pre-trained weights as is, we assign the weights parameter to the embedding values retrieved from the 'create_emb' function and make it non-trainable.

In [ ]:
model = Sequential([
    Embedding(TOP_K , 50, input_length=MAX_SEQUENCE_LENGTH,
              weights=[emb], trainable=False),
    Flatten(),
    Dense(10, activation='relu'),
    Dropout(0.7),
    Dense(1, activation='sigmoid')])

In [ ]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_1 (Embedding)      (None, 250, 50)           150000    
_________________________________________________________________
flatten_1 (Flatten)          (None, 12500)             0         
_________________________________________________________________
dense_2 (Dense)              (None, 10)                125010    
_________________________________________________________________
dropout_1 (Dropout)          (None, 10)                0         
_________________________________________________________________
dense_3 (Dense)              (None, 1)                 11        
Total params: 275,021
Trainable params: 125,021
Non-trainable params: 150,000
_________________________________________________________________


In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(x_train, train_class, validation_data=(x_val, valid_class), epochs=10, batch_size=64 )

Epoch 1/10
115/115 [==============================] - 0s 4ms/step - loss: 0.6658 - accuracy: 0.5720 - val_loss: 0.5265 - val_accuracy: 0.6961
Epoch 2/10
115/115 [==============================] - 0s 3ms/step - loss: 0.5928 - accuracy: 0.6668 - val_loss: 0.4516 - val_accuracy: 0.7953
Epoch 3/10
115/115 [==============================] - 0s 3ms/step - loss: 0.5334 - accuracy: 0.7125 - val_loss: 0.4455 - val_accuracy: 0.7694
Epoch 4/10
115/115 [==============================] - 0s 3ms/step - loss: 0.5056 - accuracy: 0.7338 - val_loss: 0.4353 - val_accuracy: 0.7759
Epoch 5/10
115/115 [==============================] - 0s 3ms/step - loss: 0.4676 - accuracy: 0.7599 - val_loss: 0.4385 - val_accuracy: 0.7565
Epoch 6/10
115/115 [==============================] - 0s 3ms/step - loss: 0.4330 - accuracy: 0.7845 - val_loss: 0.4190 - val_accuracy: 0.7845
Epoch 7/10
115/115 [==============================] - 0s 3ms/step - loss: 0.4269 - accuracy: 0.7894 - val_loss: 0.4306 - val_accuracy: 0.7931
Epoch 

### Training the embeddings

# New Section

Generally, glove would have learnt the embedding value with lots of contexts and the values could be quite generic. So, sometimes the model would not perform well on our dataset. In such cases, we could retrain the embeddings but also ensure that the learning rate is very small so that it does not abruptly change the weight and undo the complete learning.

In [ ]:
model.layers[0].trainable=True

In [ ]:
model.optimizer.lr=1e-4

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(x_train, train_class, validation_data=(x_val, valid_class), epochs=10, batch_size=256)

Epoch 1/10
29/29 [==============================] - 0s 7ms/step - loss: 0.3154 - accuracy: 0.8652 - val_loss: 0.5188 - val_accuracy: 0.8297
Epoch 2/10
29/29 [==============================] - 0s 6ms/step - loss: 0.3125 - accuracy: 0.8692 - val_loss: 0.5243 - val_accuracy: 0.8297
Epoch 3/10
29/29 [==============================] - 0s 6ms/step - loss: 0.3161 - accuracy: 0.8618 - val_loss: 0.5225 - val_accuracy: 0.8297
Epoch 4/10
29/29 [==============================] - 0s 6ms/step - loss: 0.3214 - accuracy: 0.8601 - val_loss: 0.5252 - val_accuracy: 0.8297
Epoch 5/10
29/29 [==============================] - 0s 6ms/step - loss: 0.3143 - accuracy: 0.8670 - val_loss: 0.5262 - val_accuracy: 0.8297
Epoch 6/10
29/29 [==============================] - 0s 6ms/step - loss: 0.3150 - accuracy: 0.8662 - val_loss: 0.5294 - val_accuracy: 0.8297
Epoch 7/10
29/29 [==============================] - 0s 6ms/step - loss: 0.3146 - accuracy: 0.8645 - val_loss: 0.5297 - val_accuracy: 0.8297
Epoch 8/10
29/29 [==